In [1]:

from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter

DATA_DIR = Path("../data/raw/NEU-DET")
SPLITS = ["train", "validation"]

for split in SPLITS:
    annotation_dir = DATA_DIR / split / "annotations"
    image_dir = DATA_DIR / split / "images"

    class_counts = Counter()
    objects_per_image = Counter()
    mismatched_classes = []
    multi_object_files = []

    for xml_path in annotation_dir.glob("*.xml"):
        root = ET.parse(xml_path).getroot()
        filename = root.findtext("filename", "")
        objects = root.findall("object")

        labels = [obj.findtext("name", "") for obj in objects]
        class_counts.update(labels)
        objects_per_image[len(objects)] += 1

        # Check whether the filename prefix agrees with its XML class.
        for label in labels:
            if not xml_path.stem.startswith(label + "_"):
                mismatched_classes.append(
                    (xml_path.name, label)
                )

        if len(objects) > 1:
            multi_object_files.append(
                (xml_path.name, labels)
            )

    print(f"\n--- {split.upper()} ---")
    print("Objects per class:", dict(class_counts))
    print("Objects per annotation:", dict(objects_per_image))
    print("Files with multiple objects:", len(multi_object_files))
    print("Filename/class inconsistencies:", len(mismatched_classes))

    print("\nExamples of multiple-object annotations:")
    for item in multi_object_files[:5]:
        print(item)

    print("\nExamples of filename/class inconsistencies:")
    for item in mismatched_classes[:5]:
        print(item)



--- TRAIN ---
Objects per class: {'crazing': 524, 'patches': 688, 'inclusion': 852, 'pitted_surface': 345, 'rolled-in_scale': 496, 'scratches': 427}
Objects per annotation: {1: 406, 2: 534, 4: 121, 3: 291, 5: 50, 6: 21, 7: 7, 9: 4, 8: 5}
Files with multiple objects: 1033
Filename/class inconsistencies: 160

Examples of multiple-object annotations:
('crazing_10.xml', ['crazing', 'crazing'])
('crazing_100.xml', ['crazing', 'crazing', 'crazing', 'crazing'])
('crazing_102.xml', ['crazing', 'crazing'])
('crazing_103.xml', ['crazing', 'crazing', 'crazing'])
('crazing_104.xml', ['crazing', 'crazing', 'crazing', 'patches'])

Examples of filename/class inconsistencies:
('crazing_104.xml', 'patches')
('crazing_105.xml', 'inclusion')
('crazing_112.xml', 'patches')
('crazing_152.xml', 'patches')
('crazing_20.xml', 'patches')

--- VALIDATION ---
Objects per class: {'crazing': 165, 'inclusion': 159, 'patches': 193, 'pitted_surface': 87, 'rolled-in_scale': 132, 'scratches': 121}
Objects per annotati